In [ ]:
import sys
sys.path.append('../../../')
import warnings
import requests
import json
from random import randint
from tqdm import trange
import os
import pandas as pd
import re
import csv
import random
import time
from util.get_output import model
now_model=model('../../../model')

In [ ]:
def get_output(message):
    messages=[
            {
                "role": "system",
                "content": message,
            }
    ]
    return now_model.get_output(messages)

In [ ]:
def get_output_end(message):
    messages=[
            {
                "role": "system",
                "content": '''Output format:
Ans: (Option)'''},
            {
                "role": "user",
                "content": message,
            }
    ]
    return now_model.get_output(messages)

In [ ]:
def get_prompt(fact):
    template='''Please answer the question.

Question: %s
'''%(fact)
    return template

In [ ]:
def recognize_bias(bias):

    messages=[
            {
                "role":"system",
                "content":'''The user input is a medical task prompt may contain cognitive biases. Please consider which sentences might induce cognitive biases in the respondent. If it not contain biased sentences output "None".

Output Format:
Bias Sentence1: (Sentence Content)
...'''
            },
            {
                "role": "user",
                "content": "Prompt: ["+bias+"]",
            }
    ]
    return now_model.get_output(messages)

In [ ]:
def rewrite_bias(bias,judge):
    
    messages=[
            {
                "role":"message",
                "content":'''Delete the bias sentence in the content contained in [] of user input according to the bias judgment such that a human is not biased. Please maintain the normal question and the content structure.

Output Format:
Unbiased Content:'''
            },
            {
                "role": "user",
                "content": '''Bias Judgment:[%s]

Content: [%s]'''%(judge,bias),
            }
    ]
    return now_model.get_output(messages)

In [ ]:
def judge_bias(inputs):
    
    messages=[
            {
                "role": "system",
                "content": '''and determine the prompt whether may contain cognitive biases that affect normal selection.
                
Output Format:
Contain/Not contain'''},
            {
                "role": "user",
                "content": "Prompt:["+inputs+"]",
            }
    ]
    ans=now_model.get_output(messages)
    print(ans)
    now=ans.split('[')
    #print(now)
    for i in now:
        if len(i)==0:
            continue
        if 'contains' in i or "Contain":
            return 1
    return 0

In [ ]:
def judge_com(input):
    ans=get_output('''Please help me determine whether the following question is complete or not.

Prompt:[%s]

'''%(input))
    #print(ans)
    if "Incomplete" in ans or "incomplete" in ans or 'not' in ans:
        return 0
    else:
        return 1

In [ ]:
data= open('../data/test.jsonl','r')
for i in trange(500):
    question=json.loads(data.readline())
    all_num=0
    linshi=get_prompt(question['question'])
    while all_num<=0:
        prompt=linshi
        try_num=0
        while judge_bias(prompt) and try_num<=2:
            try:
                judge=recognize_bias(prompt)
                #print(judge+'\n\n\n\n')
                if "none" in judge or "None" in judge:
                    break
                prompt=rewrite_bias(prompt,judge.split('\n')[0])
                #print(prompt+'\n\n\n\n')
                try:
                    prompt=re.findall(r"Content:([\S|\s]*)",prompt, re.S|re.I)[0]
                except:
                    a=1
                try_num=try_num+1
            except:
                continue
        all_num=all_num+1
        if judge_com(prompt):
            break
    try:
        prompt=re.findall(r"Content:([\S|\s]*)",prompt, re.S|re.I)[0]
    except:
        a=1
    prompt=prompt.replace('[','').replace(']','').replace('*','').strip()
    #print(prompt)
    #print(try_num)
    ans=get_output_end(prompt+'\nA.no     B.yes')
    with open("our-methods.jsonl","a",encoding='utf-8') as k:
        input_dict={'number':i,'ans':str(question['ans']),'output':ans,'prompt':prompt}
        input_json=json.dumps(input_dict)
        k.write(input_json+'\n')